# 07 · Machine-learning permeability, blind validation and uncertainty
**F-O gas field — Notebook 07 (core of Paper 3)**

Inputs: unzipped **F-O_composites_v6** (Notebook 06) and `core_data_shifted.csv` (from F-O_composites_v3).

The question for Paper 3 is not "can ML fit core permeability" (it always can) but **does it predict permeability in a well it
has never seen, better than the petrophysical methods of Notebooks 03 and 06, and how sure can we be?**

Steps
1. **Noise floor** — how well can any log-scale method match core plugs? (plug-to-plug scatter; calibrated NMR in F-O4)
2. **Blind comparison** (leave-one-well-out): field φ–k transform, rock-type transforms (Notebook 06), multiple linear
   regression, random forest, gradient boosting — with and without resistivity.
3. **Why random K-fold is misleading** — the same models scored with shuffled folds.
4. **What the model uses** — grouped drop-column importance.
5. **How many cored wells are needed** — learning curve over training-well subsets.
6. **Uncertainty** — well-grouped conformal P10–P90 intervals, checked for coverage in blind wells.
7. **Unsupervised electrofacies** vs core rock types.
8. **Independent check in F-O8** (no core; NMR).
9. **Final model** → composite v7 (`PERM_ML`, `PERM_ML_P10`, `PERM_ML_P90`) and blind depth plots.

In [ ]:
# Cell 1 — paths (EDIT)
from pathlib import Path
V6_DIR   = Path(r"C:\Users\YOUR_NAME\...\F-O_composites_v6")
CORE_CSV = Path(r"C:\Users\YOUR_NAME\...\F-O_composites_v3\core_data_shifted.csv")
OUT_DIR  = Path(r"C:\Users\YOUR_NAME\...\F-O_v7_outputs")
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
# Cell 2 — load logs and build the training table (one row per core plug)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, lasio, itertools, warnings
warnings.filterwarnings("ignore")
from petrotools import rocktype as rt, mlperm as ml
from petrotools.saturation import RT_CURVE
from sklearn.ensemble import RandomForestRegressor
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
WELLS = ["F-O1", "F-O2", "F-O3", "F-O4", "F-O6", "F-O8", "F-R1", "F-S1"]
WIN = {"F-O1": (3695, 3815.5), "F-O2": (3595, 3699.5), "F-O3": (3678, 3794), "F-O4": (3700, 3876), "F-O6": (3700, 3930),
       "F-O8": (3700, 3916), "F-R1": (3836, 4035), "F-S1": (3596, 3712.5)}
logs = {}
for w in WELLS:
    d = pd.read_csv(V6_DIR / f"{w}_composite_v6.csv", index_col=0)
    d["DTX"] = d["DT"].combine_first(d["DTCO"]) if {"DT", "DTCO"} <= set(d) else d.get("DT", d.get("DTCO"))   # compressional sonic, any tool
    d["LRT"] = np.log10(d[RT_CURVE[w]].clip(lower=0.1)); d["DSEP"] = d.NPHI - d.PHIT
    logs[w] = d
ROCK = ml.ROCK; ALL = ROCK + ["RHOB_USE", "DTX", "LRT"]
core = pd.read_csv(CORE_CSV); core = core[(core.kkl_md > 0.001) & (core.phi_pct > 1)].copy()
core["RT"] = rt.rock_type(rt.winland_r35(core.kkl_md, core.phi_pct).values).values
T = ml.sample_logs(core, logs, ALL)
T = T[T.BADHOLE != 1].dropna(subset=ALL + ["RT"]).reset_index(drop=True)
T["lk"] = np.log10(T.kkl_md); T["RT"] = T.RT.astype(int)
print(len(T), "plugs with complete logs in gauge hole"); T.groupby("well").size().rename("n").to_frame().T

## Step 1 — the noise floor

In [ ]:
# Cell 3 — plug-to-plug scatter (pairs <= 0.35 m apart) and calibrated NMR in F-O4
d = T.sort_values(["well", "depth_log_m"]); dz = d.depth_log_m.diff(); dl = d.lk.diff()
pair = (d.well == d.well.shift()) & (dz > 0) & (dz <= 0.35)
NUGGET = float(np.sqrt((dl[pair] ** 2).mean() / 2))
print(f"plug pairs <= 0.35 m apart: n = {pair.sum()}, single-plug scatter = {NUGGET:.2f} dex (x{10 ** NUGGET:.1f})")
f4 = logs["F-O4"]; z = f4.index.to_numpy(); e = []
for _, r in T[T.well == "F-O4"].iterrows():
    i = int(np.clip(np.searchsorted(z, r.depth_log_m), 1, len(z) - 2)); v = f4.KTIM.iloc[i - 1:i + 2].mean() * 3.32
    if v > 0: e.append(np.log10(v) - r.lk)
e = np.array(e); NMR_RMSE = float(np.sqrt((e ** 2).mean()))
print(f"F-O4 NMR KTIM x 3.32 vs core (calibrated on this core, not blind): n = {len(e)}, bias {e.mean():+.2f}, RMSE {NMR_RMSE:.2f} dex")

Neighbouring plugs a few tens of centimetres apart already differ by ~0.4 decades (one standard deviation), and a log averages
~0.6 m of rock. **No log-based method can beat ≈ 0.4–0.5 dex RMSE against single plugs.** Calibrated NMR — the best permeability
log available — reaches ≈ 0.8 dex in F-O4 even on its own calibration core. These two numbers frame every result below.

## Step 2 — blind comparison (leave-one-well-out)

In [ ]:
# Cell 4
P_rock = ml.cross_predict(T, ROCK); P_all = ml.cross_predict(T, ALL)
S_rock = ml.score(P_rock, T.lk, T.well); S_all = ml.score(P_all, T.lk, T.well)
S_rock.to_csv(OUT_DIR / "blind_scores_rock_features.csv"); S_all.to_csv(OUT_DIR / "blind_scores_with_resistivity.csv")
print("rock / pore-system features only"); display(S_rock)
print("adding RHOB, DT and log10 deep resistivity"); display(S_all[["pooled_rmse_dex", "mean_well_rmse_dex", "worst_well_rmse_dex", "F_O1_bias_dex"]])

In [ ]:
# Cell 5 — blind cross-plots
fig, ax = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
for a, c, t in zip(ax, ["FIELD", "ROCKTYPE", "RF"], ["field φ–k transform (NB03)", "rock-type transforms (NB06)", "random forest"]):
    for w, g in T.groupby("well"): a.scatter(g.lk, P_rock.loc[g.index, c], s=9, alpha=0.6, label=w)
    a.plot([-3, 3], [-3, 3], "k-", lw=1); a.plot([-3, 3], [-2, 4], "k--", lw=0.6); a.plot([-3, 3], [-4, 2], "k--", lw=0.6)
    a.set(title=f"{t}\nblind RMSE {S_rock.loc[c, 'pooled_rmse_dex']:.2f} dex", xlabel="core log10 k (mD)", xlim=(-3, 3), ylim=(-3, 3)); a.grid(alpha=0.3)
ax[0].set_ylabel("predicted log10 k (mD), well withheld"); ax[0].legend(fontsize=8)
plt.tight_layout(); fig.savefig(OUT_DIR / "fig_blind_crossplots.png", dpi=200); plt.show()

**Result.** Under blind testing the random forest (≈ 0.73 dex) and the rock-type transforms of Notebook 06 (≈ 0.74 dex) perform
the same, and both clearly beat the single field transform (≈ 0.91 dex; F-O1 under-predicted ×19). Multiple linear regression is
in between. Adding resistivity, RHOB and DT does not help the random forest; boosting removes more of the F-O1 bias but not the
overall error. **The gain comes from describing the rock (clay, cement, gas/clay separation), not from the algorithm** — a
physically-based rock-type model captures nearly all of it. Blind ML error is already as good as calibrated NMR in F-O4.

## Step 3 — why random K-fold over-states skill

In [ ]:
# Cell 6
P_kf = ml.cross_predict(T, ROCK, how="kfold"); S_kf = ml.score(P_kf, T.lk, T.well)
cmp = pd.DataFrame({"random 5-fold": S_kf.pooled_rmse_dex, "leave-one-well-out": S_rock.pooled_rmse_dex})
cmp["optimism_%"] = (100 * (1 - cmp["random 5-fold"] / cmp["leave-one-well-out"])).round(0); cmp.to_csv(OUT_DIR / "kfold_vs_lowo.csv"); display(cmp)
ax = cmp[["random 5-fold", "leave-one-well-out"]].plot.bar(figsize=(8, 4), rot=0, color=["#bbbbbb", "#1f77b4"])
ax.axhline(NUGGET, color="k", ls=":", lw=1); ax.text(4.4, NUGGET + 0.01, "plug scatter", ha="right", fontsize=8)
ax.set_ylim(0, 1.1); ax.legend(loc="upper left", ncol=2); ax.set_ylabel("RMSE (dex)"); ax.set_title("Same models, two validation schemes"); plt.tight_layout()
plt.savefig(OUT_DIR / "fig_kfold_vs_lowo.png", dpi=200); plt.show()

Shuffled folds put neighbouring plugs from the same core on both sides of the split, so the model is partly tested on samples it
has effectively seen. Every method looks 7–20 % better than it is (the flexible tree models gain most), and the field transform's F-O1 failure is
partly hidden. **Report
leave-one-well-out (or blocked) scores only** — this is a methodological point worth a paragraph in Paper 3.

## Step 4 — what the model uses

In [ ]:
# Cell 7 — grouped drop-column importance (random forest, blind). Correlated logs are dropped together, because permuting one
# member of a correlated group (e.g. PHIT with NPHI and DSEP; PHIT = NPHI - DSEP) gives misleading importances.
def lowo_rmse(F, n=200):
    e = []
    for w in T.well.unique():
        a, b = T[T.well != w], T[T.well == w]
        e.append(RandomForestRegressor(**{**ml.RF_REG, "n_estimators": n}).fit(a[F], a.lk).predict(b[F]) - b.lk)
    e = pd.concat(e); return float(np.sqrt((e ** 2).mean()))
GROUPS = {"porosity + gas/clay separation\n(PHIT, NPHI, DSEP)": ["PHIT", "NPHI", "DSEP"], "clay\n(VSH_GR, V_CLAY)": ["VSH_GR", "V_CLAY"],
          "carbonate cement\n(V_CARB, RHOMAA)": ["V_CARB", "RHOMAA"]}
base = lowo_rmse(ROCK)
imp = pd.Series({g: lowo_rmse([f for f in ROCK if f not in c]) - base for g, c in GROUPS.items()}).sort_values()
print(f"baseline blind RMSE {base:.3f} dex; increase when a group is removed:"); display(imp.round(3).to_frame("ΔRMSE_dex"))
ax = imp.plot.barh(figsize=(7, 3), color="#1f77b4"); ax.set_xlabel("increase in blind RMSE when removed (dex)"); plt.tight_layout()
plt.savefig(OUT_DIR / "fig_group_importance.png", dpi=200); plt.show()

**Clay is the controlling input** (removing VSH_GR and V_CLAY costs ≈ 0.1 dex); carbonate cement adds a little. Removing the
porosity group does *not* hurt blind skill: porosity information is still carried by RHOMAA/V_CARB (built from the neutron–sonic
porosity) and by clay, which fills pore space (r(PHIT, V_CLAY) ≈ −0.4), while porosity logs carry well-specific calibration
offsets (e.g. F-O3, +2.9 pu in Notebook 03) that transfer poorly between wells. Note that DSEP (NPHI − PHIT) mixes clay (positive)
and gas (negative): better rock holds more gas, so part of its signal is a saturation–height effect — physically real and
available wherever the model is applied, but state it.

## Step 5 — how many cored wells are needed?

In [ ]:
# Cell 8 — every subset of k training wells, scored on each withheld well (~2 min)
rows = []
for w in T.well.unique():
    others = [v for v in T.well.unique() if v != w]; b = T[T.well == w]
    for k in range(1, len(others) + 1):
        for comb in itertools.combinations(others, k):
            a = T[T.well.isin(comb)]
            e_rf = RandomForestRegressor(**{**ml.RF_REG, "n_estimators": 200}).fit(a[ROCK], a.lk).predict(b[ROCK]) - b.lk
            e_f = np.polyval(np.polyfit(100 * a.PHIT, a.lk, 1), 100 * b.PHIT) - b.lk
            rows.append(dict(test=w, k=k, RF=np.sqrt((e_rf ** 2).mean()), FIELD=np.sqrt((e_f ** 2).mean())))
lc = pd.DataFrame(rows); lc.to_csv(OUT_DIR / "learning_curve.csv", index=False)
s = lc.groupby("k")[["RF", "FIELD"]].agg(["median", lambda x: x.quantile(0.9)]); s.columns = ["RF_median", "RF_P90", "FIELD_median", "FIELD_P90"]
display(s.round(2))
fig, ax = plt.subplots(figsize=(7, 4))
for c, col in [("RF", "#1f77b4"), ("FIELD", "#888888")]:
    ax.plot(s.index, s[f"{c}_median"], "o-", color=col, label=f"{c} median"); ax.plot(s.index, s[f"{c}_P90"], "o--", color=col, label=f"{c} P90 (bad case)")
ax.axhline(NUGGET, color="k", ls=":", lw=1); ax.set(xlabel="number of cored training wells", ylabel="blind RMSE (dex)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); fig.savefig(OUT_DIR / "fig_learning_curve.png", dpi=200); plt.show()

Error keeps falling up to the five training wells available, and the bad-case (P90) error falls fastest: more cored wells mainly
protect against an unrepresentative well such as F-O1. The curve has not flattened, so additional core (or sidewall cores in
F-O8/F-R1) would still improve the model.

## Step 6 — uncertainty: well-grouped conformal intervals

In [ ]:
# Cell 9 — for each withheld well, the interval comes only from the other wells' blind residuals (nested), then coverage is checked
cov = []
for w in T.well.unique():
    a, b = T[T.well != w], T[T.well == w]
    lo, hi = ml.well_conformal(a, ROCK)
    p = RandomForestRegressor(**ml.RF_REG).fit(a[ROCK], a.lk).predict(b[ROCK])
    cov.append(dict(well=w, n=len(b), coverage_P10_P90=((b.lk >= p - hi) & (b.lk <= p - lo)).mean(), width_dex=hi - lo))
cov = pd.DataFrame(cov).round(2); cov.to_csv(OUT_DIR / "conformal_coverage.csv", index=False); display(cov)
print(f"mean coverage {cov.coverage_P10_P90.mean():.2f} (target 0.80); mean P10-P90 width {cov.width_dex.mean():.2f} dex "
      f"(x{10 ** cov.width_dex.mean():.0f} between P10 and P90)")

Intervals from ordinary quantile regression (trained on pooled plugs) cover only ~50 % of blind-well plugs, because they ignore
the well-to-well shift. Building the interval from **blind-well residuals** gives the intended ~80 % coverage on average — but the
honest P10–P90 range is wide (~1.9 dex, about ×80). That is the true uncertainty of log-derived permeability in an uncored well
here and belongs in the volumetric / deliverability uncertainty, not hidden.

## Step 7 — unsupervised electrofacies vs core rock types

In [ ]:
# Cell 10
EF = ["VSH_GR", "PHIT", "V_CARB", "RHOMAA", "DSEP"]; X = StandardScaler().fit_transform(T[EF]); ef = []
for k in range(2, 8):
    g = GaussianMixture(k, covariance_type="full", random_state=0, n_init=5).fit(X)
    ef.append(dict(k=k, BIC=round(g.bic(X)), ARI_vs_core_RT=round(adjusted_rand_score(T.RT, g.predict(X)), 3)))
pd.DataFrame(ef)

Clusters of log responses do not reproduce the r35 rock types (adjusted Rand index ≈ 0.03, ~0 = random). Logs group rock by
lithology and cement; pore-throat size needs core to define it. **Supervised** rock typing (Notebook 06) or direct regression is
required — a useful negative result for the paper.

## Step 8 — independent check in F-O8 (no core)

In [ ]:
# Cell 11 — models trained on all cored wells vs NMR KTIM x 3.32 in F-O8 clean sand
RF_ALL = RandomForestRegressor(**ml.RF_REG).fit(T[ROCK], T.lk); bF = np.polyfit(100 * T.PHIT, T.lk, 1)
d = logs["F-O8"].loc[WIN["F-O8"][0]:WIN["F-O8"][1]]
ok = d[ROCK].notna().all(axis=1) & (d.BADHOLE != 1) & (d.VSH_GR < 0.4) & (d.KTIM > 0); nm = np.log10(d.loc[ok, "KTIM"] * 3.32)
chk = []
for n, p in [("FIELD", np.polyval(bF, 100 * d.loc[ok, "PHIT"])), ("ROCKTYPE (PERM_RT)", np.log10(d.loc[ok, "PERM_RT"])),
             ("RF", RF_ALL.predict(d.loc[ok, ROCK]))]:
    e = p - nm; chk.append(dict(method=n, n=int(ok.sum()), bias_dex=e.mean(), rmse_dex=np.sqrt((e ** 2).mean()), r=np.corrcoef(p, nm)[0, 1]))
pd.DataFrame(chk).round(2)

All three methods agree with NMR in F-O8 only to ~1.2–1.3 dex and sit ~0.3 dex below it. In a gas leg NMR permeability is itself
uncertain (low hydrogen index, gas T2 signal) and its ×3.32 factor comes from one well, so F-O8 cannot discriminate between methods
— report it as a consistency check, not a validation.

## Step 9 — final model, composite v7 and blind depth plots

In [ ]:
# Cell 12 — final random forest on all cored plugs; P10/P90 from blind residuals of all six wells
LO, HI = ml.well_conformal(T, ROCK, n_estimators=400)
print(f"interval offsets (dex): P10 = pred - {HI:.2f}, P90 = pred + {-LO:.2f}")
NEWU = dict(PERM_ML="mD", PERM_ML_P10="mD", PERM_ML_P90="mD")
for w in WELLS:
    d = logs[w].copy(); ok = d[ROCK].notna().all(axis=1) & (d.index >= WIN[w][0] - 30) & (d.index <= WIN[w][1] + 30)
    for c in NEWU: d[c] = np.nan
    if ok.sum():
        p = RF_ALL.predict(d.loc[ok, ROCK])
        d.loc[ok, "PERM_ML"] = 10 ** p; d.loc[ok, "PERM_ML_P10"] = 10 ** (p - HI); d.loc[ok, "PERM_ML_P90"] = 10 ** (p - LO)
    d = d.drop(columns=["LRT", "DSEP", "DTX"])
    d.to_csv(OUT_DIR / f"{w}_composite_v7.csv", float_format="%.5f")
    units = {c.mnemonic: c.unit for c in lasio.read(str(V6_DIR / f"{w}_composite_v6.las")).curves}; units.update(NEWU)
    las = lasio.LASFile(); las.well["WELL"].value = w; las.well["FLD"].value = "F-O Gas Field"
    las.other = (f"{w} composite v7 (Notebook 07). PERM_ML random forest on rock/pore-system logs (PHIT VSH_GR V_CLAY V_CARB RHOMAA "
                 f"NPHI DSEP) trained on 617 core plugs; P10/P90 well-grouped conformal (blind-well residuals).")
    las.append_curve("DEPT", d.index.to_numpy(), unit="M")
    for c in d.columns: las.append_curve(c, d[c].to_numpy(), unit=units.get(c, ""))
    las.write(str(OUT_DIR / f"{w}_composite_v7.las"), version=2.0, fmt="%.5f")
rows = []
for w in WELLS:
    s = pd.read_csv(OUT_DIR / f"{w}_composite_v7.csv", index_col=0).loc[WIN[w][0]:WIN[w][1]]; s = s[s.VSH_GR < 0.4]
    rows.append(dict(well=w, PERM_median_mD=s.PERM.median(), PERM_RT_median_mD=s.PERM_RT.median(), PERM_ML_median_mD=s.PERM_ML.median(),
                     PERM_ML_P10_median=s.PERM_ML_P10.median(), PERM_ML_P90_median=s.PERM_ML_P90.median()))
summ = pd.DataFrame(rows).round(3); summ.to_csv(OUT_DIR / "perm_summary_v7.csv", index=False); summ

In [ ]:
# Cell 13 — blind depth plots (the plotted well is withheld from training) for F-O1 and F-O4
fig, ax = plt.subplots(1, 2, figsize=(10, 9))
for a, w in zip(ax, ["F-O1", "F-O4"]):
    tr = T[T.well != w]; rf = RandomForestRegressor(**ml.RF_REG).fit(tr[ROCK], tr.lk); lo, hi = ml.well_conformal(tr, ROCK)
    bF = np.polyfit(100 * tr.PHIT, tr.lk, 1)
    d = logs[w].loc[WIN[w][0]:min(WIN[w][1], WIN[w][0] + 120)]; ok = d[ROCK].notna().all(axis=1); z = d.index
    p = pd.Series(np.nan, index=z); p[ok] = rf.predict(d.loc[ok, ROCK]); p = p.to_numpy()   # gaps stay blank
    a.fill_betweenx(z, 10 ** (p - hi), 10 ** (p - lo), color="#1f77b4", alpha=0.2, label="RF P10–P90 (blind)")
    a.plot(10 ** p, z, color="#1f77b4", lw=1, label="RF (blind)")
    a.plot(10 ** np.polyval(bF, 100 * d.PHIT.where(ok)), z, color="#888888", lw=1, label="field transform (blind)")
    c = T[T.well == w]; a.scatter(c.kkl_md, c.depth_log_m, s=10, color="k", zorder=3, label="core (Klinkenberg)")
    a.set(xscale="log", xlim=(1e-3, 1e3), title=w, xlabel="permeability (mD)"); a.invert_yaxis(); a.grid(alpha=0.3, which="both")
ax[0].set_ylabel("depth (m, log depth)"); ax[0].legend(fontsize=8, loc="lower left")
plt.tight_layout(); fig.savefig(OUT_DIR / "fig_blind_depth_FO1_FO4.png", dpi=200); plt.show()

## Notes for Paper 3
* **Validation design is the headline:** leave-one-well-out throughout; show the random-K-fold optimism (Step 3).
* **Benchmarks, not just models:** plug scatter (≈ 0.4 dex) and calibrated NMR (≈ 0.8 dex) bound what is achievable; ML blind
  (≈ 0.73 dex) is at NMR level and ≈ 20 % better than the field transform, but not better than physically-based rock typing (NB06).
* **Uncertainty must be well-grouped**; pooled quantile regression under-covers badly (~50 % for a nominal 80 %).
* **Limitations:** six cored wells (617 plugs); mineral inputs calibrated to one XRD well; ambient vs in-situ permeability
  (Klinkenberg, no overburden correction); log-to-plug scale mismatch.
* **Possible extension** (the earlier plan): seismic impedance as an extra feature to map permeability between wells — needs the
  3-D seismic volume and well ties.